In [2]:
import yfinance as yf
import pandas as pd
import os
import requests
from bs4 import BeautifulSoup
import re
from io import StringIO
import datetime

base_dir = "data"
date_end = datetime.datetime.now().strftime("%Y-%m-%d")
date_start = (datetime.datetime.now() - datetime.timedelta(days=365*10)).strftime("%Y-%m-%d")

## Get SP500 tickers from Wikipedia, download stock price data for these tickers

In [ ]:
"https://gpwbenchmark.pl/karta-indeksu?isin=PL9999999375"

In [ ]:
def fetch_tickers_from_wikipedia(url, table_index=0, column_name="Symbol"):
    """Fetch tickers from a Wikipedia table using a custom User-Agent."""
    headers = {
        "User-Agent": "my-ticker-fetch"
    }

    try:
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()

        tables = pd.read_html(StringIO(response.text))
        return tables[table_index][column_name].dropna().tolist()

    except Exception as e:
        print(f"Error fetching tickers from {url}: {e}")
        return []

index_info = { 
        "url": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        "table_index": 0,
        "column": "Symbol",
        "suffix": ""
    }

print(f"\n Downloading data for S&P 500 components")

tickers = fetch_tickers_from_wikipedia(index_info["url"], index_info["table_index"], index_info["column"])
tickers = [str(t).strip() for t in tickers]
print(tickers)
pd.Series(tickers).to_csv(os.path.join(base_dir, "sp500_tickers.csv"), index=False)

for ticker in tickers:
    full_ticker = ticker + index_info["suffix"]
    try:
        df = yf.download(full_ticker, start=date_start, end=date_end, interval="1d", auto_adjust=True)
        if not df.empty:
            df.to_csv(os.path.join(base_dir, f"sp500/{ticker}.csv"))
            print(f"Saved: {ticker}")
    except Exception as e:
        print(f"{ticker}: {e}")

## Get polish stock data

In [ ]:
# tickers from WIG30 index as of 2026-04-05

mapping = {
    "PKNORLEN": "PKN",
    "PKOBP": "PKO",
    "PEKAO": "PEO",
    "KGHM": "KGH",
    "PZU": "PZU",
    "LPP": "LPP",
    "SANPL": "SPL",
    "ALLEGRO": "ALE",
    "CDPROJEKT": "CDR",
    "DINOPL": "DNP",
    "MBANK": "MBK",
    "TAURONPE": "TPE",
    "ALIOR": "ALR",
    "MILLENNIUM": "MIL",
    "KETY": "KTY",
    "ASSECOPOL": "ACP",
    "PGE": "PGE",
    "ORANGEPL": "OPL",
    "BUDIMEX": "BDX",
    "KRUK": "KRU",
    "XTB": "XTB",
    "ENEA": "ENA",
    "CYFRPLSAT": "CPS",
    "JSW": "JSW",
}

index_dir = os.path.join(base_dir, "wig30")
os.makedirs(index_dir, exist_ok=True)

for name, ticker in mapping.items():
    try:
        df = yf.download(ticker+".WA", start=date_start, end=date_end, interval="1d", auto_adjust=True)
        if not df.empty:
            df.to_csv(os.path.join(index_dir, f"{name}.csv"))
            print(f"Saved: {name}")
    except Exception as e:
        print(f"{ticker}: {e}")